# Simulacion en RocketPy
This notebook was generated using Rocket-Serializer, a RocketPy tool to convert simulation files to RocketPy simulations
El notebook fue generado usando el siguiente archivo de parametros: `.\notebooks\simulacion_cohete.ipynb\parameters.json`


In [ ]:
from rocketpy import Environment, SolidMotor, Rocket, Flight, TrapezoidalFins, EllipticalFins, RailButtons, NoseCone, Tail, Parachute
import datetime


## Entorno
Configuración operativa con coordenadas, elevación base y modelo GFS activo.


In [ ]:
from rocketpy import Environment
import datetime
import requests
import math

# ==========================================
# 1. BLOQUE DE CONFIGURACION (EXTRAIDO DEL .ORK)
# ==========================================
# Estas coordenadas fueron extraidas directamente de tu archivo OpenRocket
LATITUD = 45.0
LONGITUD = 0.0
ELEVACION = 0.0

# ==========================================
# 2. INICIALIZACION GEOGRAFICA EXACTA (t=0)
# ==========================================
# Respeto absoluto a la causalidad fisica: El punto de lanzamiento es estrictamente el origen (t=0).
env = Environment()
env.set_location(LATITUD, LONGITUD)
env.set_elevation(ELEVACION)

try:
    # Calculo de componentes de viento a partir de los datos de OpenRocket
    wind_speed = 2.0
    wind_dir_rad = 1.5707963267948966 # Radianes desde el Norte
    # En meteorologia (y OpenRocket), la direccion es de donde VIENE el viento.
    # Para RocketPy, wind_u es hacia el Este (+X), wind_v es hacia el Norte (+Y).
    wind_u = -wind_speed * math.sin(wind_dir_rad)
    wind_v = -wind_speed * math.cos(wind_dir_rad)
    
    # OpenRocket esta configurado para usar Atmosfera Estandar (ISA)
    env.set_atmospheric_model(type='standard_atmosphere')
    # RocketPy requiere llamar a custom_atmosphere para inyectar viento sobre ISA
    env.set_atmospheric_model(
        type='custom_atmosphere',
        wind_u=wind_u,
        wind_v=wind_v
    )
    print(f'[OK] Entorno configurado: Atmosfera Estandar (ISA) y viento del .ork ({wind_speed:.2f} m/s).')

except Exception as e:
    print(f'[ADVERTENCIA] Hubo un error al configurar la atmosfera: {e}')
    env.set_atmospheric_model(type='standard_atmosphere')

env.all_info()


## Motor
Currently, only Solid Motors are supported by Rocket-Serializer. If you want to use a Liquid/Hybrid motor, please use rocketpy directly.


In [ ]:
motor = SolidMotor(
    thrust_source='thrust_source.csv',
    dry_mass=0,
    dry_inertia=[0.001, 0.001, 0.001],
    center_of_dry_mass_position=0,
    grains_center_of_mass_position=0,
    grain_number=1,
    grain_density=224.5572389303639,
    grain_outer_radius=0.009,
    grain_initial_inner_radius=0.0045,
    grain_initial_height=0.07,
    grain_separation=0,
    nozzle_radius=0.006749999999999999,
    nozzle_position=0.035,
    throat_radius=0.0045,
    reshape_thrust_curve=False,  # Not implemented in Rocket-Serializer
    interpolation_method='linear',
    coordinate_system_orientation='combustion_chamber_to_nozzle',
)


In [ ]:
motor.all_info()


## Cohete
Currently, only single stage rockets are supported by Rocket-Serializer
We will start by defining the aerodynamic surfaces, and then build the rocket.


### Ojivas (Nosecones)


In [ ]:
nosecone = NoseCone(
    length=0.1,
    kind='ogive',
    base_radius=0.0125,
    rocket_radius=0.0125,
    name='0.1',
)


### Aletas (Fins)
As rocketpy allows for multiple fins sets, we will create a dictionary with all the fins sets and then add them to the rocket


In [ ]:
trapezoidal_fins = {}


In [ ]:
trapezoidal_fins[0] = TrapezoidalFins(
    n=3,
    root_chord=0.0508,
    tip_chord=0.0508,
    span=0.03,
    cant_angle=0.0,
    sweep_length= 0.0254,
    sweep_angle= None,
    rocket_radius=0.0125,
    name='Trapezoidal fin set',
)



### Transiciones / Colas (Tails)
As rocketpy allows for multiple tails, we will create a dictionary with all the tails and then add them to the rocket


In [ ]:
tails = {}


### Paracaidas
As rocketpy allows for multiple parachutes, we will create a dictionary with all the parachutes and then add them to the rocket


In [ ]:
parachutes = {}


In [ ]:
parachutes = {}
parachutes[0] = Parachute(
    name='Parachute',
    cd_s=0.071,
    trigger='apogee',
    sampling_rate=100, 
)


In [ ]:
rocket = Rocket(
    radius=0.0125,
    mass=0.062,
    inertia=[0.000974, 0.000974, 9.895e-06],
    power_off_drag='drag_curve.csv',
    power_on_drag='drag_curve.csv',
    center_of_mass_without_motor=0.242,
    coordinate_system_orientation='nose_to_tail',
)


### Agregando superficies al cohete
Ahora que tenemos todas las superficies, podemos agregarlas al cohete


In [ ]:
# ============================================================
# CCTE: Extracción Geométrica Dinámica del .ork
# Calcula posiciones absolutas en tiempo de ejecución.
# ============================================================
import sys, os

# Resolver ruta raíz robustamente (buscando la carpeta 'scripts')
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, 'scripts')) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)

sys.path.append(os.path.join(_root, 'scripts'))
from ork_geometry import OrkGeometry

_ork_path = os.path.join(_root, 'disenos', 'cohete.ork')
if os.path.exists(_ork_path):
    geo = OrkGeometry(_ork_path)
    FIN_POSITION_ABSOLUTE = geo.first_fin_set.fin_leading_edge if geo.first_fin_set else 0.0
    print(f'Geometría dinámica cargada. Aletas en: {FIN_POSITION_ABSOLUTE:.4f}m desde la nariz')
else:
    print('Advertencia: cohete.ork no encontrado.')
    FIN_POSITION_ABSOLUTE = 0.0


In [ ]:
rocket.add_surfaces(surfaces=[nosecone, trapezoidal_fins[0]], positions=[0.0, FIN_POSITION_ABSOLUTE])

In [ ]:
rocket.add_motor(motor, position= 0.37199999999999933)


Agregando paracaidas al cohete


In [ ]:
rocket.parachutes = list(parachutes.values())


### Botones de Riel (Rail Buttons)


In [ ]:
# No se agregaron botones de riel al cohete.

In [ ]:
### Informacion del Cohete
rocket.all_info()


## Flight
We will now create the flight simulation. Let's go!


In [ ]:
flight = Flight(
    rocket=rocket,
    environment=env,
    rail_length=1.0,
    inclination=90.0,
    heading=90.0,
    terminate_on_apogee=False,
    max_time=600,
)

In [ ]:
import io
from contextlib import redirect_stdout

# Capturar salida de flight.all_info()
f_out = io.StringIO()
with redirect_stdout(f_out):
    flight.all_info()

out_str = f_out.getvalue()
print(out_str)  # Mantener salida visible en consola

# Guardar silenciosamente en archivo de texto en la raiz del proyecto
with open('../../reporte_mision_urgente.txt', 'w', encoding='utf-8') as f:
    f.write(out_str)
print('\n[SISTEMA] Reporte tecnico guardado exitosamente en reporte_mision_urgente.txt')


## Comparacion de Resultados
We will now compare the results of the simulation with the parameters used to create it. Let's go!


In [ ]:
### Parametros: OpenRocket vs RocketPy
time_to_apogee_ork = 3.476
time_to_apogee_rpy = flight.apogee_time
print(f"Tiempo hasta el apogeo (OpenRocket): {time_to_apogee_ork:.3f} s")
print(f"Tiempo hasta el apogeo (RocketPy):   {time_to_apogee_rpy:.3f} s")
apogee_difference = time_to_apogee_rpy - time_to_apogee_ork
error = abs((apogee_difference)/ (time_to_apogee_rpy if time_to_apogee_rpy != 0 else 1e-9)) * 100
print(f"Tiempo hasta el apogeo (diferencia):   {error:.3f} %")
print()

flight_time_ork = 15.849
flight_time_rpy = flight.t_final
print(f"Tiempo de vuelo (OpenRocket): {flight_time_ork:.3f} s")
print(f"Tiempo de vuelo (RocketPy):   {flight_time_rpy:.3f} s")
flight_time_difference = flight_time_rpy - flight_time_ork
error_flight_time = abs((flight_time_difference)/ (flight_time_rpy if flight_time_rpy != 0 else 1e-9)) * 100
print(f"Tiempo de vuelo (diferencia):   {error_flight_time:.3f} %")
print()

ground_hit_velocity_ork = -4.633
ground_hit_velocity_rpy = flight.impact_velocity
print(f"Velocidad de impacto (OpenRocket): {ground_hit_velocity_ork:.3f} m/s")
print(f"Velocidad de impacto (RocketPy):   {ground_hit_velocity_rpy:.3f} m/s")
ground_hit_velocity_difference = ground_hit_velocity_rpy - ground_hit_velocity_ork
error_ground_hit_velocity = abs((ground_hit_velocity_difference)/ (ground_hit_velocity_rpy if ground_hit_velocity_rpy != 0 else 1e-9)) * 100
print(f"Velocidad de impacto (diferencia):   {error_ground_hit_velocity:.3f} %")
print()

launch_rod_velocity_ork = 15.365
launch_rod_velocity_rpy = flight.out_of_rail_velocity
print(f"Velocidad de salida del riel (OpenRocket): {launch_rod_velocity_ork:.3f} m/s")
print(f"Velocidad de salida del riel (RocketPy):   {launch_rod_velocity_rpy:.3f} m/s")
launch_rod_velocity_difference = launch_rod_velocity_rpy - launch_rod_velocity_ork
error_launch_rod_velocity = abs((launch_rod_velocity_difference)/ (launch_rod_velocity_rpy if launch_rod_velocity_rpy != 0 else 1e-9)) * 100
print(f"Velocidad de salida del riel (diferencia):   {error_launch_rod_velocity:.3f} %")
print()

max_acceleration_ork = 143.641
max_acceleration_rpy = flight.max_acceleration
print(f"Aceleracion maxima (OpenRocket): {max_acceleration_ork:.3f} m/s²")
print(f"Aceleracion maxima (RocketPy):   {max_acceleration_rpy:.3f} m/s²")
max_acceleration_difference = max_acceleration_rpy - max_acceleration_ork
error_max_acceleration = abs((max_acceleration_difference)/ (max_acceleration_rpy if max_acceleration_rpy != 0 else 1e-9)) * 100
print(f"Aceleracion maxima (diferencia):   {error_max_acceleration:.3f} %")
print()

max_altitude_ork = 50.436
max_altitude_rpy = flight.apogee - flight.env.elevation
print(f"Altitud maxima (OpenRocket): {max_altitude_ork:.3f} m")
print(f"Altitud maxima (RocketPy):   {max_altitude_rpy:.3f} m")
max_altitude_difference = max_altitude_rpy - max_altitude_ork
error_max_altitude = abs((max_altitude_difference)/ (max_altitude_rpy if max_altitude_rpy != 0 else 1e-9)) * 100
print(f"Altitud maxima (diferencia):   {error_max_altitude:.3f} %")
print()

max_mach_ork = 0.086
max_mach_rpy = flight.max_mach_number 
print(f"Mach maximo (OpenRocket): {max_mach_ork:.3f}")
print(f"Mach maximo (RocketPy):   {max_mach_rpy:.3f}")
max_mach_difference = max_mach_rpy - max_mach_ork
error_max_mach = abs((max_mach_difference)/ (max_mach_rpy if max_mach_rpy != 0 else 1e-9)) * 100
print(f"Mach maximo (diferencia):   {error_max_mach:.3f} %")
print()

max_velocity_ork = 29.209
max_velocity_rpy = flight.max_speed
print(f"Velocidad maxima (OpenRocket): {max_velocity_ork:.3f} m/s")
print(f"Velocidad maxima (RocketPy):   {max_velocity_rpy:.3f} m/s")
max_velocity_difference = max_velocity_rpy - max_velocity_ork
error_max_velocity = abs((max_velocity_difference)/ (max_velocity_rpy if max_velocity_rpy != 0 else 1e-9)) * 100
print(f"Velocidad maxima (diferencia):   {error_max_velocity:.3f} %")
print()

max_thrust_ork = 9.73
max_thrust_rpy = flight.rocket.motor.thrust.max
print(f"Empuje maximo (OpenRocket): {max_thrust_ork:.3f} N")
print(f"Empuje maximo (RocketPy):   {max_thrust_rpy:.3f} N")
max_thrust_difference = max_thrust_rpy - max_thrust_ork
error_max_thrust = abs((max_thrust_difference)/ (max_thrust_rpy if max_thrust_rpy != 0 else 1e-9)) * 100
print(f"Empuje maximo (diferencia):   {error_max_thrust:.3f} %")
print()

burnout_stability_margin_ork = 2.946
burnout_stability_margin_rpy = flight.stability_margin(flight.rocket.motor.burn_out_time)
print(f"Margen de estabilidad (Burnout) (OpenRocket): {burnout_stability_margin_ork:.3f}")
print(f"Margen de estabilidad (Burnout) (RocketPy):   {burnout_stability_margin_rpy:.3f}")
burnout_stability_margin_difference = burnout_stability_margin_rpy - burnout_stability_margin_ork
error_burnout_stability_margin = abs((burnout_stability_margin_difference)/ (burnout_stability_margin_rpy if burnout_stability_margin_rpy != 0 else 1e-9)) * 100
print(f"Margen de estabilidad (Burnout) (diferencia):   {error_burnout_stability_margin:.3f} %")
print()

max_stability_margin_ork = 3.302
max_stability_margin_rpy = flight.max_stability_margin
print(f"Margen de estabilidad maximo (OpenRocket): {max_stability_margin_ork:.3f}")
print(f"Margen de estabilidad maximo (RocketPy):   {max_stability_margin_rpy:.3f}")
max_stability_margin_difference = max_stability_margin_rpy - max_stability_margin_ork
error_max_stability_margin = abs((max_stability_margin_difference)/ (max_stability_margin_rpy if max_stability_margin_rpy != 0 else 1e-9)) * 100
print(f"Margen de estabilidad maximo (diferencia):   {error_max_stability_margin:.3f} %")
print()

min_stability_margin_ork = 0.0
min_stability_margin_rpy = flight.min_stability_margin
print(f"Margen de estabilidad minimo (OpenRocket): {min_stability_margin_ork:.3f}")
print(f"Margen de estabilidad minimo (RocketPy):   {min_stability_margin_rpy:.3f}")
min_stability_margin_difference = min_stability_margin_rpy - min_stability_margin_ork
error_min_stability_margin = abs((min_stability_margin_difference)/ (min_stability_margin_rpy if min_stability_margin_rpy != 0 else 1e-9)) * 100
print(f"Margen de estabilidad minimo (diferencia):   {error_min_stability_margin:.3f} %")
print()



## Visualizacion y Telemetria
Visualizacion en 3D de la trayectoria y exportacion logistica.


In [ ]:
flight.info()
flight.plots.linear_kinematics_data()
flight.plots.trajectory_3d()

# ==========================================
# MODULO DE RECUPERACION Y TELEMETRIA
# ==========================================
import os
from rocketpy.simulation.flight_data_exporter import FlightDataExporter

# Asegurar existencia del directorio de resultados en la raiz del proyecto
os.makedirs('../../resultados', exist_ok=True)

# 1. Extraccion de coordenadas
impact_lat = flight.latitude(flight.t_final)
impact_lon = flight.longitude(flight.t_final)
print("\n[SISTEMA] --- REPORTE DE NAVEGACION ---")
print(f"-> PUNTO DE INICIO (Rampa)     : {LATITUD:.6f}, {LONGITUD:.6f}  <-- Coordenadas ingresadas")
print(f"-> PUNTO DE FINAL (Aterrizaje) : {impact_lat:.6f}, {impact_lon:.6f}")
print(f"-> Deriva Balistica Total      : {math.sqrt(flight.x_impact**2 + flight.y_impact**2):.2f} metros")

# Instanciar exportador (RocketPy v1.12+)
exporter = FlightDataExporter(flight)

# 2. Exportacion KML para Google Earth
exporter.export_kml(
    file_name="../../resultados/trayectoria_operativa.kml",
    extrude=True,
    altitude_mode="relative_to_ground"
)
print("[SISTEMA] Archivo KML generado en /resultados para visualizacion espacial.")

# 3. Exportacion de la matriz balistica a CSV
exporter.export_data(
    "../../resultados/datos_telemetria_simulada.csv",
    "z", "vz", "az", "mach_number"
)
print("[SISTEMA] Matriz de vuelo exportada a CSV en /resultados con exito.")

# 4. Reporte de Recuperacion
print('\n[SISTEMA] --- REPORTE DE RECUPERACION ---')
for event in flight.parachute_events:
    trigger_time = event[0]
    parachute = event[1]
    trigger_z = flight.z(trigger_time)
    trigger_agl = trigger_z - env.elevation
    print(f'Paracaidas: {parachute.name} | Tiempo: {trigger_time:.2f} s | Altitud AGL: {trigger_agl:.2f} m')
print('-------------------------------------------')
